# OSCA Chapter 2 Plotting Scripts

This notebook aggregates all plotting and visualization scripts for Chapter 2. Note: R scripts are prefixed with `%%R` so they can be run in Python using `rpy2` magic (`%load_ext rpy2.ipython`).

## 02_plot_tree_bio.py

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from Bio import Phylo
import os

tree_file = "/home/mfcaval/github/doutorado/analyses/02_phylogeny/results/iqtree.renamed.treefile"
out_pdf = "/home/mfcaval/github/doutorado/reports/figures/02_phylogeny/OSCA_phylogenetic_tree.pdf"
out_png = "/home/mfcaval/github/doutorado/reports/figures/02_phylogeny/OSCA_phylogenetic_tree.png"

def get_color(name):
    if not name:
        return 'black'
    if 'OSCA1' in name or 'AT1G32090' in name or 'AT4G02900' in name or 'AT4G04340' in name or 'AT4G22120' in name or 'AT1G62320' in name or 'AT1G11960' in name or 'AT4G15430' in name or 'AT3G21620' in name:
        return '#1f77b4'
    if 'OSCA2' in name or 'AT3G54510' in name or 'AT1G58520' in name or 'AT1G10090' in name or 'AT1G69450' in name or 'AT3G01100' in name:
        return '#ff7f0e'
    if 'OSCA3' in name or 'AT1G30360' in name:
        return '#2ca02c'
    if 'OSCA4' in name or 'AT4G35870' in name:
        return '#d62728'
    return 'black'

def label_colors(label):
    return get_color(label)

tree = Phylo.read(tree_file, "newick")
tree.ladderize()

fig, ax = plt.subplots(figsize=(12, 16))

# We can customize the drawing by passing label_colors dictionary or function in newer biopython, 
# but biopython Phylo.draw label_colors takes a dict or a func mapping labels to colors.
Phylo.draw(tree, do_show=False, axes=ax, label_func=lambda x: x.name if x.name else "", label_colors=label_colors, show_confidence=False)

# Make the lines gray so labels pop out
for collection in ax.collections:
    collection.set_color('#333333')
    collection.set_linewidth(1.0)

os.makedirs(os.path.dirname(out_pdf), exist_ok=True)
plt.savefig(out_pdf, bbox_inches='tight', dpi=300)
plt.savefig(out_png, bbox_inches='tight', dpi=300)
print("Phylogenetic tree saved successfully via Biopython!")


## 02_plot_tree.py

In [ ]:
import os
from ete3 import Tree, TreeStyle, NodeStyle, TextFace

tree_file = "/home/mfcaval/github/doutorado/analyses/02_phylogeny/results/iqtree.renamed.treefile"
out_pdf = "/home/mfcaval/github/doutorado/reports/figures/02_phylogeny/OSCA_phylogenetic_tree.pdf"
out_png = "/home/mfcaval/github/doutorado/reports/figures/02_phylogeny/OSCA_phylogenetic_tree.png"

# Colors for clades
# OSCA1: Blue (#1f77b4), OSCA2: Orange (#ff7f0e), OSCA3: Green (#2ca02c), OSCA4: Red (#d62728)
def get_color(name):
    if not name:
        return 'black'
    if 'OSCA1' in name or 'AT1' in name or 'AT4G04340' in name or 'AT4G22120' in name or 'AT1G62320' in name or 'AT1G11960' in name or 'AT4G15430' in name or 'AT3G21620' in name or 'AT4G02900' in name or 'AT1G32090' in name:
        return '#1f77b4'
    if 'OSCA2' in name or 'AT3G54510' in name or 'AT1G58520' in name or 'AT1G10090' in name or 'AT1G69450' in name or 'AT3G01100' in name:
        return '#ff7f0e'
    if 'OSCA3' in name or 'AT1G30360' in name:
        return '#2ca02c'
    if 'OSCA4' in name or 'AT4G35870' in name:
        return '#d62728'
    return 'black'

# Load tree
t = Tree(tree_file)
t.ladderize()

# Create a TreeStyle
ts = TreeStyle()
ts.show_leaf_name = False
ts.mode = "r" # rectangular
ts.scale = 120 # pixel per branch length unit

# Iterate over all nodes
for n in t.traverse():
    ns = NodeStyle()
    
    # Hide internal nodes visually
    if not n.is_leaf():
        ns["size"] = 0
    else:
        ns["size"] = 0
        color = get_color(n.name)
        
        # Add the leaf name as a TextFace with the clade color
        face = TextFace(n.name, fsize=10, fgcolor=color)
        face.margin_left = 5
        n.add_face(face, column=0, position="branch-right")
        
    n.set_style(ns)

# Ensure output directory exists
os.makedirs(os.path.dirname(out_pdf), exist_ok=True)

# Save
t.render(out_pdf, w=800, units="px", tree_style=ts)
t.render(out_png, w=800, units="px", tree_style=ts)
print("Phylogenetic tree saved successfully!")


## 04_plot_heatmaps.py

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
import argparse

# Common functional categorization for PlantCARE motifs
MOTIF_CATEGORIES = {
    'Hormone': [
        'ABRE', 'CGTCA-motif', 'TGACG-motif', 'TCA-element', 
        'GARE-motif', 'P-box', 'TGA-element', 'AuxRR-core', 'TATC-box', 'SARE'
    ],
    'Stress': [
        'MBS', 'TC-rich repeats', 'LTR', 'ARE', 'W box', 'STRE', 
        'WUN-motif', 'MRE', 'GC-motif', 'DRE core', 'DRE1'
    ],
    'Light': [
        'G-box', 'G-Box', 'Sp1', 'Box 4', 'GT1-motif', 'AE-box',
        'I-box', 'GATA-motif', 'Box II', 'Box III', 'TCT-motif', 'ACE', 
        'LAMP-element', 'ATCT-motif'
    ],
    'Development & Other': [
        'O2-site', 'CAT-box', 'RY-element', 'MBSI', 'HD-Zip 3', 
        'GCN4_motif', 'A-box', 'circadian', 'CAAAT'
    ]
}

def clean_data(df):
    """
    Remove ubiquitous elements and parsing artifacts
    """
    cols_to_drop = [c for c in df.columns if c in ['TATA-box', 'CAAT-box', 'TATA'] or 'Unnamed' in c]
    df_clean = df.drop(columns=cols_to_drop, errors='ignore')
    return df_clean

def plot_heatmap(df, species, output_dir):
    # Ensure index is Gene
    if 'Gene' in df.columns:
        df = df.set_index('Gene')
    
    df_clean = clean_data(df)
    
    # Identify motifs that are in our dictionary and present in the dataframe
    selected_motifs = []
    category_colors = []
    
    # We will map colors to categories for annotation
    cat_palette = sns.color_palette("Set2", len(MOTIF_CATEGORIES))
    cat_color_map = {cat: cat_palette[i] for i, cat in enumerate(MOTIF_CATEGORIES.keys())}
    
    col_colors_dict = {}
    for cat, motifs in MOTIF_CATEGORIES.items():
        for m in motifs:
            if m in df_clean.columns:
                selected_motifs.append(m)
                col_colors_dict[m] = cat_color_map[cat]
                
    # Filter df to only known categorized motifs for cleaner visualization
    if selected_motifs:
        df_plot = df_clean[selected_motifs]
    else:
        df_plot = df_clean # Fallback if none matched
        
    # Remove genes (rows) that have 0 motifs across all selected motifs
    df_plot = df_plot.loc[(df_plot != 0).any(axis=1)]
    
    # Create column colors series for seaborn
    col_colors = pd.Series(col_colors_dict).reindex(df_plot.columns)
    
    plt.figure(figsize=(14, 10))
    g = sns.clustermap(
        df_plot, 
        cmap="YlOrRd", 
        linewidths=1, 
        linecolor='white',
        col_colors=col_colors if selected_motifs else None,
        col_cluster=True,
        row_cluster=True,
        figsize=(12, 8),
        cbar_kws={'label': 'Number of occurrences'}
    )
    
    g.fig.suptitle(f'Cis-acting Elements in {species} OSCA Promoters', y=1.05, fontsize=16)
    
    # Adjust rotation
    plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right')
    plt.setp(g.ax_heatmap.get_yticklabels(), rotation=0)
    
    # Create custom legend for categories
    if selected_motifs:
        from matplotlib.patches import Patch
        legend_elements = [Patch(facecolor=cat_color_map[cat], label=cat) for cat in MOTIF_CATEGORIES.keys()]
        g.ax_heatmap.legend(handles=legend_elements, loc='upper left', bbox_to_anchor=(1.05, 1.05), title='Motif Category')
        
    plt.tight_layout()
    
    output_path = os.path.join(output_dir, f"{species}_promoter_heatmap.png")
    g.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Heatmap saved to {output_path}")

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input_dir", default="../results", help="Directory with _motif_counts.tsv files")
    parser.add_argument("--output_dir", default="../results", help="Directory to save the plots")
    args = parser.parse_args()
    
    for species in ["Vper", "Vtub"]:
        file_path = os.path.join(args.input_dir, f"{species}_motif_counts.tsv")
        if os.path.exists(file_path):
            print(f"Plotting for {species}...")
            df = pd.read_csv(file_path, sep='\t')
            plot_heatmap(df, species, args.output_dir)
        else:
            print(f"File not found: {file_path}")

if __name__ == "__main__":
    main()


## 02_plot_tree_circular.R

In [ ]:
%%R

library(ggtree)
library(treeio)
library(ggplot2)

tree <- read.newick("results/iqtree.renamed.treefile")

tips <- tree$tip.label
groupInfo <- list(
  OSCA1 = tips[grepl("OSCA1|AT1G32090|AT4G02900|AT4G04340|AT4G22120|AT1G62320|AT1G11960|AT4G15430|AT3G21620", tips)],
  OSCA2 = tips[grepl("OSCA2|AT3G54510|AT1G58520|AT1G10090|AT1G69450|AT3G01100", tips)],
  OSCA3 = tips[grepl("OSCA3|AT1G30360", tips)],
  OSCA4 = tips[grepl("OSCA4|AT4G35870", tips)]
)
tree <- groupOTU(tree, groupInfo)

p <- ggtree(tree, layout="circular", aes(color=group), size=1.0) + 
     geom_tiplab(size=3.5, offset=0.05) + 
     scale_color_manual(values=c("OSCA1"="#1f77b4", "OSCA2"="#ff7f0e", "OSCA3"="#2ca02c", "OSCA4"="#d62728", "0"="black")) +
     theme(legend.position="right")

dir.create("../../reports/figures/02_phylogeny", showWarnings = FALSE, recursive = TRUE)
ggsave("../../reports/figures/02_phylogeny/OSCA_phylogenetic_tree_circular.pdf", p, width = 10, height = 10)
ggsave("../../reports/figures/02_phylogeny/OSCA_phylogenetic_tree_circular.png", p, width = 10, height = 10, dpi=300)
print("Circular tree saved successfully!")


## 02_plot_tree.R

In [ ]:
%%R

library(ggtree)
library(treeio)
library(ggplot2)

tree <- read.newick("results/iqtree.renamed.treefile")

# Create a group vector based on tips
tips <- tree$tip.label
groupInfo <- list(
  OSCA1 = tips[grepl("OSCA1|AT1G32090|AT4G02900|AT4G04340|AT4G22120|AT1G62320|AT1G11960|AT4G15430|AT3G21620", tips)],
  OSCA2 = tips[grepl("OSCA2|AT3G54510|AT1G58520|AT1G10090|AT1G69450|AT3G01100", tips)],
  OSCA3 = tips[grepl("OSCA3|AT1G30360", tips)],
  OSCA4 = tips[grepl("OSCA4|AT4G35870", tips)]
)
tree <- groupOTU(tree, groupInfo)

p <- ggtree(tree, aes(color=group), size=0.8) + 
     geom_tiplab(size=3) + 
     scale_color_manual(values=c("OSCA1"="#1f77b4", "OSCA2"="#ff7f0e", "OSCA3"="#2ca02c", "OSCA4"="#d62728", "0"="black")) +
     theme_tree2() +
     xlim(0, 5)

dir.create("../../reports/figures/02_phylogeny", showWarnings = FALSE, recursive = TRUE)
ggsave("../../reports/figures/02_phylogeny/OSCA_phylogenetic_tree.pdf", p, width = 8, height = 10)
ggsave("../../reports/figures/02_phylogeny/OSCA_phylogenetic_tree.png", p, width = 8, height = 10, dpi=300)
print("Saved ggtree")


## 04_plot_contrast_fel.R

In [ ]:
%%R

library(ggplot2)
library(dplyr)

# Data extracted from contrast_fel_summary.tsv
data <- data.frame(
  Clade = c(rep("OSCA1", 5), rep("OSCA2", 11), rep("OSCA3", 1)),
  Site = c(
    98, 181, 252, 710, 973, 
    8, 71, 76, 147, 191, 308, 394, 395, 432, 596, 720, 
    593
  ),
  Length = c(
    rep(1029, 5),
    rep(889, 11),
    rep(904, 1)
  )
)

# Set Clade as factor with desired order
data$Clade <- factor(data$Clade, levels = c("OSCA1", "OSCA2", "OSCA3"))

# Create the Lollipop plot
p <- ggplot(data) +
  # Protein backbone (alignment length)
  geom_segment(aes(x = 1, xend = Length, y = 0, yend = 0), color = "grey70", size = 3, lineend = "round") +
  # Lollipops
  geom_segment(aes(x = Site, xend = Site, y = 0, yend = 1, color = Clade), size = 1.2) +
  geom_point(aes(x = Site, y = 1, color = Clade), size = 4) +
  # Add site labels on top of the lollipops
  geom_text(aes(x = Site, y = 1.2, label = Site, color = Clade), size = 3.5, fontface = "bold", angle = 45, hjust = 0) +
  facet_wrap(~ Clade, ncol = 1, strip.position = "right") +
  theme_minimal() +
  scale_y_continuous(limits = c(-0.5, 2.0), breaks = NULL) +
  scale_color_manual(values = c("OSCA1" = "#1b9e77", "OSCA2" = "#d95f02", "OSCA3" = "#7570b3")) +
  labs(title = "Contrast-FEL: Significant Sites of Selective Divergence",
       subtitle = "Sites with significantly different evolutionary rates between V. peripherica and V. tubiflora",
       x = "Codon Position (Alignment)",
       y = NULL) +
  theme(
    plot.title = element_text(face = "bold", size = 18, hjust = 0.5),
    plot.subtitle = element_text(size = 12, hjust = 0.5, color = "grey40", margin = margin(b = 20)),
    axis.title.x = element_text(face = "bold", size = 12, margin = margin(t = 15)),
    strip.text = element_text(face = "bold", size = 14),
    panel.grid = element_blank(),
    legend.position = "none",
    plot.margin = margin(20, 20, 20, 20)
  )

# Output paths
output_png <- "/home/mfcaval/github/doutorado/analyses/04_evolutionary_rates/results/contrast_fel_sites.png"
output_svg <- "/home/mfcaval/github/doutorado/analyses/04_evolutionary_rates/results/contrast_fel_sites.svg"

# Save plots
ggsave(output_png, p, width = 12, height = 7, dpi = 300, bg = "white")
ggsave(output_svg, p, width = 12, height = 7, bg = "white")

cat("Plots saved to:\n", output_png, "\n", output_svg, "\n")


# Main Manuscript Figures

The following scripts generate the final composite figures for the manuscript.

## plot_figure_1_osca_phylogeny.py

In [ ]:
#!/usr/bin/env python3
"""
Figure 1: Maximum-Likelihood Phylogeny and Pfam Domain Architecture of the OSCA Family in Vellozia
Panel A: Circular ML Phylogeny (IQ-TREE 2) of 76 OSCA proteins across 6 angiosperm genomes
Panel B: Structural Domain Architecture (Pfam) of the 14 Vellozia peripherica OSCA subunits
Panel C: Structural Domain Architecture (Pfam) of the 13 Vellozia tubiflora OSCA subunits
"""

import os
import subprocess
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image, ImageChops

# Global Typography & Style
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 8
plt.rcParams['axes.labelsize'] = 8.5
plt.rcParams['axes.titlesize'] = 9.5

SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_DIR = os.path.dirname(os.path.dirname(SCRIPT_DIR))
OUT_DIR = os.path.join(PROJECT_DIR, "figures")
INTER_DIR = os.path.join(OUT_DIR, "intermediates")
os.makedirs(INTER_DIR, exist_ok=True)

def trim(im):
    bg = Image.new(im.mode, im.size, (255, 255, 255, 255) if im.mode == "RGBA" else (255, 255, 255))
    diff = ImageChops.difference(im, bg)
    bbox = diff.getbbox()
    if bbox:
        w, h = im.size
        return im.crop((max(0, bbox[0] - 15), max(0, bbox[1] - 15), min(w, bbox[2] + 15), min(h, bbox[3] + 15)))
    return im

def plot_figure_1():
    svg_tree = os.path.join(OUT_DIR, "oscatree.svg")
    svg_vper = os.path.join(OUT_DIR, "vper_osca_domains.svg")
    svg_vtub = os.path.join(OUT_DIR, "vtub_osca_domains.svg")

    uhd_tree = os.path.join(INTER_DIR, "oscatree_uhd.png")
    uhd_vper = os.path.join(INTER_DIR, "vper_domains_uhd.png")
    uhd_vtub = os.path.join(INTER_DIR, "vtub_domains_uhd.png")

    # High-resolution rasterization via rsvg-convert (zoom=4 -> crisp 300-600 DPI equivalent)
    if not os.path.exists(uhd_tree):
        subprocess.run(["rsvg-convert", "--zoom=4", svg_tree, "-o", uhd_tree], check=True)
    if not os.path.exists(uhd_vper):
        subprocess.run(["rsvg-convert", "--zoom=4", svg_vper, "-o", uhd_vper], check=True)
    if not os.path.exists(uhd_vtub):
        subprocess.run(["rsvg-convert", "--zoom=4", svg_vtub, "-o", uhd_vtub], check=True)

    im_tree = Image.open(uhd_tree)
    im_vper = trim(Image.open(uhd_vper))
    im_vtub = trim(Image.open(uhd_vtub))

    # Grid layout: 2 columns (Tree on left occupying full height; Vper and Vtub stacked on right)
    fig = plt.figure(figsize=(12.0, 7.8), dpi=300)
    gs = fig.add_gridspec(2, 2, width_ratios=[1.08, 1.0], height_ratios=[1.0, 1.0],
                          wspace=0.10, hspace=0.18, left=0.035, right=0.98, top=0.95, bottom=0.03)

    ax_tree = fig.add_subplot(gs[:, 0])
    ax_vper = fig.add_subplot(gs[0, 1])
    ax_vtub = fig.add_subplot(gs[1, 1])

    ax_tree.imshow(im_tree)
    ax_tree.axis('off')

    ax_vper.imshow(im_vper)
    ax_vper.axis('off')

    ax_vtub.imshow(im_vtub)
    ax_vtub.axis('off')

    fig.canvas.draw()
    r = fig.canvas.get_renderer()

    # Panel A: Tree
    bb_a = ax_tree.get_window_extent(r)
    fbb_a = fig.transFigure.inverted().transform(bb_a)
    fig.text(fbb_a[0][0], fbb_a[1][1] + 0.012, "A   Maximum-Likelihood Phylogeny of the OSCA Family", 
             fontsize=9.2, fontweight='bold', va='bottom', ha='left')

    # Panels B & C: Domain Architectures
    bb_b = ax_vper.get_window_extent(r)
    fbb_b = fig.transFigure.inverted().transform(bb_b)
    bb_c = ax_vtub.get_window_extent(r)
    fbb_c = fig.transFigure.inverted().transform(bb_c)

    # Strictly align B and C title labels along the left margin of the right column
    x_right_col = min(fbb_b[0][0], fbb_c[0][0])
    fig.text(x_right_col, fbb_b[1][1] + 0.012, "B   Vellozia peripherica OSCA Domain Architecture", 
             fontsize=9.2, fontweight='bold', va='bottom', ha='left')
    fig.text(x_right_col, fbb_c[1][1] + 0.012, "C   Vellozia tubiflora OSCA Domain Architecture", 
             fontsize=9.2, fontweight='bold', va='bottom', ha='left')

    out_pdf = os.path.join(OUT_DIR, "Figure_1_OSCA_Phylogeny_and_Architecture.pdf")
    out_png = os.path.join(OUT_DIR, "Figure_1_OSCA_Phylogeny_and_Architecture.png")

    plt.savefig(out_pdf, dpi=300, bbox_inches='tight')
    plt.savefig(out_png, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Figure 1 re-saved successfully:\n  - {out_pdf}\n  - {out_png}")

if __name__ == "__main__":
    plot_figure_1()


## plot_figure_2_osca_synteny.py

In [ ]:
#!/usr/bin/env python3
"""
Figure 2: Comparative Synteny and Local Microsyntenic Dynamics of the OSCA Family in Vellozia
Panel A: Whole-Genome Macrosynteny Collinearity across Homologous Chromosomes (Chr1–Chr9)
Panel B: Microsynteny of the Conserved Tandem Cluster (OSCA1.3, OSCA1.4, OSCA1.5) on Chromosome 7
Panel C: Microsynteny of the Lineage-Specific Tandem Duplication of VpOSCA2.4a/b on Chromosome 1
"""

import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image, ImageChops
import numpy as np

# Global Typography & Style
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 8
plt.rcParams['axes.labelsize'] = 8.5
plt.rcParams['axes.titlesize'] = 9.5

SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_DIR = os.path.dirname(os.path.dirname(SCRIPT_DIR))
OUT_DIR = os.path.join(PROJECT_DIR, "figures")
INTER_DIR = os.path.join(OUT_DIR, "intermediates")
os.makedirs(INTER_DIR, exist_ok=True)

def trim(im):
    bg = Image.new(im.mode, im.size, (255, 255, 255))
    diff = ImageChops.difference(im, bg)
    bbox = diff.getbbox()
    if bbox:
        w, h = im.size
        left = max(0, bbox[0] - 15)
        top = max(0, bbox[1] - 15)
        right = min(w, bbox[2] + 15)
        bottom = min(h, bbox[3] + 15)
        return im.crop((left, top, right, bottom))
    return im

def plot_figure_2():
    path_k = os.path.join(INTER_DIR, "karyotype_trimmed.png")
    path_c = os.path.join(INTER_DIR, "cluster_trimmed.png")
    path_t = os.path.join(INTER_DIR, "tandem_trimmed.png")
    
    # Ensure intermediate trimmed images exist
    if not os.path.exists(path_k):
        raw_k = os.path.join(INTER_DIR, "karyotype_thick-1.png")
        if not os.path.exists(raw_k):
            os.system(f"pdftoppm -png -r 300 {os.path.join(PROJECT_DIR, '03_synteny/intergenomic/results/Vper_Vtub_osca_orthologs_karyotype.pdf')} {os.path.join(INTER_DIR, 'karyotype_thick')}")
        trim(Image.open(raw_k).convert('RGB')).save(path_k)

    if not os.path.exists(path_c):
        raw_c = os.path.join(INTER_DIR, "cluster_chr7A-1.png")
        if not os.path.exists(raw_c):
            os.system(f"pdftoppm -png -r 300 {os.path.join(PROJECT_DIR, '03_synteny/results/microsynteny_plots/osca_cluster.pdf')} {os.path.join(INTER_DIR, 'cluster_chr7A')}")
        trim(Image.open(raw_c).convert('RGB')).save(path_c)

    if not os.path.exists(path_t):
        raw_t = os.path.join(INTER_DIR, "tandem_chr1A-1.png")
        if not os.path.exists(raw_t):
            os.system(f"pdftoppm -png -r 300 {os.path.join(PROJECT_DIR, '03_synteny/results/microsynteny_plots/osca24_chr1A.pdf')} {os.path.join(INTER_DIR, 'tandem_chr1A')}")
        trim(Image.open(raw_t).convert('RGB')).save(path_t)

    im_k = Image.open(path_k)
    im_c = Image.open(path_c)
    im_t = Image.open(path_t)

    fig = plt.figure(figsize=(7.6, 9.6), dpi=300)
    gs = fig.add_gridspec(3, 1, height_ratios=[1.15, 1.05, 1.05], hspace=0.42,
                          left=0.06, right=0.96, top=0.95, bottom=0.05)

    ax_a = fig.add_subplot(gs[0, 0])
    ax_b = fig.add_subplot(gs[1, 0])
    ax_c = fig.add_subplot(gs[2, 0])

    # --- Panel A: Macrosynteny ---
    ax_a.imshow(im_k)
    ax_a.axis('off')
    
    # Legend below Panel A
    leg_patch_osca = patches.Patch(facecolor='#b84347', label='Syntenic OSCA ortholog anchors (Chr1, 4, 5, 7, 8, 9)')
    leg_patch_other = patches.Patch(facecolor='#cccccc', label='Syntenic collinear background blocks')
    ax_a.legend(handles=[leg_patch_osca, leg_patch_other], loc='upper right', frameon=True,
                facecolor='white', edgecolor='#e0e0e0', fontsize=6.8, bbox_to_anchor=(0.99, -0.05), ncol=2)

    # --- Panel B: Microsynteny Cluster Chr7A ---
    ax_b.imshow(im_c)
    ax_b.axis('off')

    # --- Panel C: Microsynteny Tandem Chr1A ---
    ax_c.imshow(im_t)
    ax_c.axis('off')

    # Aligned Panel Titles (A, B, C) strictly aligned horizontally across the figure
    fig.canvas.draw()
    r = fig.canvas.get_renderer()
    for name, ax, title in [
        ("A", ax_a, "Interspecific Macrosynteny and OSCA Collinear Distribution across Chromosomes 1–9"),
        ("B", ax_b, "Conserved OSCA1.3 / OSCA1.4 / OSCA1.5 Region on Chromosome 7"),
        ("C", ax_c, "Local OSCA2.4 Copy-Number Difference on Chromosome 1")
    ]:
        bb = ax.get_window_extent(r)
        fbb = fig.transFigure.inverted().transform(bb)
        y_top = fbb[1][1]
        fig.text(0.06, y_top + 0.015, f"{name}   {title}", fontsize=8.8, fontweight='bold', va='bottom', ha='left')

    out_pdf = os.path.join(OUT_DIR, "Figure_2_OSCA_Comparative_Synteny.pdf")
    out_png = os.path.join(OUT_DIR, "Figure_2_OSCA_Comparative_Synteny.png")
    plt.savefig(out_pdf, dpi=300, bbox_inches='tight')
    plt.savefig(out_png, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Figure 2 re-saved successfully:\n  - {out_pdf}\n  - {out_png}")

if __name__ == "__main__":
    plot_figure_2()


## plot_figure_3_osca_selection.py

In [ ]:
#!/usr/bin/env python3
"""
Figure 3: Molecular Evolutionary Rates and HyPhy Selection Dynamics of the OSCA Family in Vellozia
Panel A: Selection Intensity Parameter (K) across OSCA Clades (HyPhy RELAX: Vtub vs. Vper)
Panel B: Pairwise Selection Constraint (Ka/Ks) across 14 Orthologs and the VpOSCA2.4a/b Paralog Pair
Panel C: Site-Specific Evolutionary Divergence across Alignment Codon Positions (HyPhy Contrast-FEL)
"""

import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
import numpy as np

# Global Typography & Style
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 7.5
plt.rcParams['axes.labelsize'] = 8
plt.rcParams['axes.titlesize'] = 9
plt.rcParams['xtick.labelsize'] = 7
plt.rcParams['ytick.labelsize'] = 7

SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_DIR = os.path.dirname(os.path.dirname(SCRIPT_DIR))
OUT_DIR = os.path.join(PROJECT_DIR, "figures")
os.makedirs(OUT_DIR, exist_ok=True)

# -------------------------------------------------------------
# Data Definition
# -------------------------------------------------------------
# 1. HyPhy RELAX Results
RELAX_DATA = [
    {"Clade": "Global", "K": 0.706408557, "p": 0.000337322, "Color": "#525252", "Sig": "**"},
    {"Clade": "OSCA1", "K": 0.926177965, "p": 0.608049990, "Color": "#0072B2", "Sig": "ns"},
    {"Clade": "OSCA2", "K": 0.819142659, "p": 0.344631173, "Color": "#E69F00", "Sig": "ns"},
    # OSCA3-only rerun; OSCA4 excluded. Source: 03_synteny/results/hyphy_osca3_only_20261026/relax_summary.tsv
    {"Clade": "OSCA3", "K": 0.256532164, "p": 0.074910417, "Color": "#009E73", "Sig": "ns"},
]

# 2. Ka/Ks Data (PAL2NAL + YN00)
KAKS_CSV = os.path.join(PROJECT_DIR, "04_evolutionary_rates", "results_pal2nal_yn00", "aggregated_kaks_pal2nal_yn00.csv")

# 3. Contrast-FEL Sites
CONTRAST_SITES = {
    "OSCA1": {"sites": [51, 109, 179, 266, 520, 720], "len": 761, "color": "#0072B2"},
    "OSCA2": {"sites": [4, 65, 70, 139, 164, 209, 294, 295, 309, 402, 514], "len": 601, "color": "#E69F00"},
    "OSCA3": {"sites": [480], "len": 733, "color": "#009E73"}
}

CLADE_COLORS = {
    'OSCA1': '#1f78b4',
    'OSCA2': '#e31a1c',
    'OSCA3': '#33a02c',
    'OSCA4': '#6a3d9a'
}

def get_gene_clade(pair_name):
    if "OSCA1" in pair_name: return "OSCA1"
    if "OSCA2" in pair_name: return "OSCA2"
    if "OSCA3" in pair_name: return "OSCA3"
    if "OSCA4" in pair_name: return "OSCA4"
    return "Other"

def plot_figure_3():
    fig = plt.figure(figsize=(7.4, 7.2), dpi=300)
    gs = fig.add_gridspec(2, 2, height_ratios=[1.15, 0.95], width_ratios=[0.9, 1.3],
                          hspace=0.38, wspace=0.32,
                          left=0.08, right=0.96, top=0.94, bottom=0.07)
    fig = plt.figure(figsize=(8.5, 4.8), dpi=300)
    gs = fig.add_gridspec(1, 2, width_ratios=[0.9, 1.35], wspace=0.35,
                          left=0.09, right=0.95, top=0.86, bottom=0.14)

    ax_relax = fig.add_subplot(gs[0, 0])
    ax_kaks = fig.add_subplot(gs[0, 1])
    ax_cfel = fig.add_subplot(gs[1, :])

    # ---------------------------------------------------------
    # Panel A: HyPhy RELAX Selection Intensity (K)
    # ---------------------------------------------------------
    clades = [d["Clade"] for d in RELAX_DATA]
    k_vals = [d["K"] for d in RELAX_DATA]
    colors = [d["Color"] for d in RELAX_DATA]
    sigs = [d["Sig"] for d in RELAX_DATA]
    x_pos = np.arange(len(clades))

    bars = ax_relax.bar(x_pos, k_vals, color=colors, width=0.55, edgecolor='none', alpha=0.9)
    ax_relax.axhline(1.0, color='#666666', linestyle='--', lw=1.0, zorder=2)
    ax_relax.text(len(clades) - 0.5, 1.02, "K = 1.0 (Neutral / Equal Selection)", ha='right', va='bottom',
                  fontsize=5.8, color='#555555', style='italic')
                  fontsize=6.0, color='#555555', style='italic')

    # Add labels and significance markers
    for i, (bar, k_val, sig) in enumerate(zip(bars, k_vals, sigs)):
        y_text = bar.get_height() + 0.03
        ax_relax.text(bar.get_x() + bar.get_width()/2.0, y_text, f"{k_val:.2f}\n({sig})",
                      ha='center', va='bottom', fontsize=6.5, fontweight='bold')
                      ha='center', va='bottom', fontsize=6.8, fontweight='bold')

    ax_relax.set_xticks(x_pos)
    ax_relax.set_xticklabels(clades, fontweight='bold', fontsize=7.2)
    ax_relax.set_ylabel("Selection Intensity Parameter (K)", fontsize=7.5)
    ax_relax.set_xticklabels(clades, fontweight='bold', fontsize=7.5)
    ax_relax.set_ylabel("Selection Intensity Parameter (K)", fontsize=8.0)
    ax_relax.set_ylim(0, 1.28)
    ax_relax.spines['top'].set_visible(False)
    ax_relax.spines['right'].set_visible(False)
    ax_relax.text(-0.15, 1.06, "A", transform=ax_relax.transAxes, fontsize=11, fontweight='bold', va='top')
    ax_relax.set_title("HyPhy RELAX: Vtub vs. Vper", fontsize=8.2, fontweight='bold', pad=10, loc='left')
    ax_relax.set_title("HyPhy RELAX: Vtub vs. Vper", fontsize=8.8, fontweight='bold', pad=10, loc='left')

    # Annotation badge explaining K < 1
    badge_a = dict(boxstyle="round,pad=0.2", fc="#f7f7f7", ec="#cccccc", lw=0.5)
    ax_relax.text(0.5, 0.15, "K < 1 indicates relaxed selection in V. tubiflora\n** p < 0.01; ns = not significant",
                  transform=ax_relax.transAxes, ha='center', fontsize=5.8, color='#444444', bbox=badge_a)
    badge_a = dict(boxstyle="round,pad=0.3", fc="#f7f7f7", ec="#cccccc", lw=0.5)
    ax_relax.text(0.5, 0.12, "K < 1 indicates relaxed selection in V. tubiflora\n** p < 0.01; ns = not significant",
                  transform=ax_relax.transAxes, ha='center', fontsize=6.2, color='#444444', bbox=badge_a)

    # ---------------------------------------------------------
    # Panel B: Pairwise Ka/Ks Ratios (PAL2NAL + YN00)
    # ---------------------------------------------------------
    df_kaks = pd.read_csv(KAKS_CSV)
    df_kaks['Clade'] = df_kaks['Pair_Name'].apply(get_gene_clade)
    df_kaks['Is_Paralog'] = df_kaks['Pair_Name'].str.contains("VpOSCA2.4a_VpOSCA2.4b")
    
    # Clean pair labels: VpOSCA1.1_VtOSCA1.1 -> OSCA1.1
    def clean_pair_label(p):
        if "VpOSCA2.4a_VpOSCA2.4b" in p:
            return "Vp2.4a / Vp2.4b (Paralog)"
        parts = p.split('_')
        p1 = parts[0].replace("VpOSCA", "")
        p2 = parts[1].replace("VtOSCA", "")
        if p1 == p2:
            return f"OSCA{p1}"
        return f"Vp{p1} / Vt{p2}"

    df_kaks['Label'] = df_kaks['Pair_Name'].apply(clean_pair_label)
    df_kaks = df_kaks.sort_values('Ka/Ks', ascending=True).reset_index(drop=True)

    y_kaks = np.arange(len(df_kaks))
    for idx, row in df_kaks.iterrows():
        c_col = CLADE_COLORS.get(row['Clade'], '#555555')
        is_para = row['Is_Paralog']
        val = row['Ka/Ks']
        
        # Line from 0 to value
        ax_kaks.plot([0, val], [idx, idx], color=c_col, lw=1.2, alpha=0.8)
        # Dot at tip
        marker = 'D' if is_para else 'o'
        mface = 'white' if is_para else c_col
        ax_kaks.plot(val, idx, marker=marker, markersize=4.8, color=c_col, 
        ax_kaks.plot(val, idx, marker=marker, markersize=5.0, color=c_col, 
                     markerfacecolor=mface, markeredgewidth=1.2, zorder=3)
        # Text value
        ax_kaks.text(val + 0.02, idx, f"{val:.3f}", va='center', ha='left', fontsize=5.8, color='#333333')
        ax_kaks.text(val + 0.02, idx, f"{val:.3f}", va='center', ha='left', fontsize=6.0, color='#333333')

    ax_kaks.axvline(1.0, color='#999999', linestyle=':', lw=0.9)
    ax_kaks.axvline(0.4082, color='#e6550d', linestyle='--', lw=0.9)
    ax_kaks.text(0.42, 0.5, "Mean ω = 0.408", color='#e6550d', fontsize=5.8, style='italic')
    ax_kaks.text(0.42, 0.5, "Mean ω = 0.408", color='#e6550d', fontsize=6.2, style='italic')

    ax_kaks.set_yticks(y_kaks)
    ax_kaks.set_yticklabels(df_kaks['Label'], fontsize=6.5)
    ax_kaks.set_xlabel("Purifying Selection Ratio (ω = Ka/Ks)", fontsize=7.5)
    ax_kaks.set_yticklabels(df_kaks['Label'], fontsize=6.8)
    ax_kaks.set_xlabel("Purifying Selection Ratio (ω = Ka/Ks)", fontsize=8.0)
    ax_kaks.set_xlim(0, 0.95)
    ax_kaks.set_ylim(-0.8, len(df_kaks) - 0.2)
    ax_kaks.spines['top'].set_visible(False)
    ax_kaks.spines['right'].set_visible(False)
    ax_kaks.text(-0.25, 1.06, "B", transform=ax_kaks.transAxes, fontsize=11, fontweight='bold', va='top')
    ax_kaks.set_title("Pairwise rates (PAL2NAL + YN00)", fontsize=8.2, fontweight='bold', pad=10, loc='left')
    ax_kaks.set_title("Pairwise rates (PAL2NAL + YN00)", fontsize=8.8, fontweight='bold', pad=10, loc='left')

    # ---------------------------------------------------------
    # Panel C: Contrast-FEL Significant Codon Sites
    # ---------------------------------------------------------
    clade_order = ["OSCA1", "OSCA2", "OSCA3"]
    y_offsets = [2.0, 1.0, 0.0]
    fig.suptitle("Figure 3. Selection dynamics and pairwise evolutionary rates across OSCA clades",
                 fontsize=10.5, fontweight='bold', y=0.97, x=0.09, ha='left')

    for c_idx, c_name in enumerate(clade_order):
        c_data = CONTRAST_SITES[c_name]
        y_base = y_offsets[c_idx]
        col = c_data["color"]
        length = c_data["len"]
        sites = c_data["sites"]

        # Backbone line
        ax_cfel.plot([0, length], [y_base, y_base], color='#b0b0b0', lw=2.5, solid_capstyle='round')
        ax_cfel.text(-35, y_base, f"{c_name}", va='center', ha='right', fontsize=7.2, fontweight='bold', color=col)
        ax_cfel.text(length + 15, y_base, f"{length} codons", va='center', ha='left', fontsize=5.8, color='#666666')

        # Sites lollipops
        for s in sites:
            # Line up
            ax_cfel.plot([s, s], [y_base, y_base + 0.32], color=col, lw=0.9)
            # Dot
            ax_cfel.plot(s, y_base + 0.32, marker='o', markersize=3.8, color=col, zorder=4)
            # Site number text (angled)
            ax_cfel.text(s, y_base + 0.36, str(s), ha='center', va='bottom', fontsize=5.0, 
                         color=col, fontweight='bold', rotation=45)

    ax_cfel.set_xlim(-80, 1120)
    ax_cfel.set_ylim(-0.35, 2.7)
    ax_cfel.set_xlabel("Codon Position (Multiple Sequence Alignment)", fontsize=7.5)
    ax_cfel.get_yaxis().set_visible(False)
    ax_cfel.spines['top'].set_visible(False)
    ax_cfel.spines['right'].set_visible(False)
    ax_cfel.spines['left'].set_visible(False)
    ax_cfel.text(-0.06, 1.08, "C", transform=ax_cfel.transAxes, fontsize=11, fontweight='bold', va='top')
    ax_cfel.set_title("Contrast-FEL nominal candidates (none passed FDR q ≤ 0.20)", 
                      fontsize=8.2, fontweight='bold', pad=14, loc='left')

    # Save
    out_pdf = os.path.join(OUT_DIR, "Figure_3_OSCA_Evolutionary_Rates_and_Selection.pdf")
    out_png = os.path.join(OUT_DIR, "Figure_3_OSCA_Evolutionary_Rates_and_Selection.png")
    plt.savefig(out_pdf, dpi=300, bbox_inches='tight')
    plt.savefig(out_png, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Figure 3 saved successfully:\n  - {out_pdf}\n  - {out_png}")

if __name__ == "__main__":
    plot_figure_3()


## plot_figure_4_osca_expression.py

In [ ]:
#!/usr/bin/env python3
"""Figure 4: OSCA Baseline Tissue Expression in Vellozia peripherica and V. tubiflora."""
from pathlib import Path
import os
os.environ.setdefault('MPLCONFIGDIR', '/tmp/osca_matplotlib')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / 'figures'
plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 8,
    'axes.titlesize': 9.5,
    'axes.labelsize': 8.5,
    'xtick.labelsize': 7.5,
    'ytick.labelsize': 7.5,
    'pdf.fonttype': 42,
    'ps.fonttype': 42
})

def main():
    fig = plt.figure(figsize=(10.5, 6.6))
    gs = fig.add_gridspec(1, 2, width_ratios=[1, 1], wspace=0.32,
                          left=0.12, right=0.88, top=0.78, bottom=0.18)

    for i, species in enumerate(['Vper', 'Vtub']):
        counts = pd.read_csv(ROOT / f'06_expression/results_genecounts/{species}_OSCA_normalized_counts.csv', index_col=0)
        de = pd.read_csv(ROOT / f'06_expression/results_genecounts/{species}_OSCA_DE_results.csv')
        cols = [c for suffix in 'FCR' for c in counts.columns if c.endswith(suffix)]
        ax = fig.add_subplot(gs[0, i])
        im = ax.imshow(np.log2(counts[cols] + 1), aspect='auto', cmap='YlGnBu', vmin=0, vmax=13)

        labels = []
        for gene in counts.index:
            tests = de[de.OSCA_Name.eq(gene)].set_index('Contrast')
            marks = (' *' if tests.loc['Root_vs_Leaf', 'padj'] < .05 else '') + \
                    (' †' if tests.loc['Stem_vs_Leaf', 'padj'] < .05 else '')
            labels.append(gene + marks)

        ax.set_yticks(range(len(labels)))
        ax.set_yticklabels(labels, fontsize=7.5)
        ax.set_xticks(range(len(cols)))
        ax.set_xticklabels(cols, rotation=30, ha='right', fontsize=7.5)

        # Subtle white gridlines between cells matching Figure 6
        ax.set_xticks(np.arange(len(cols) + 1) - 0.5, minor=True)
        ax.set_yticks(np.arange(len(labels) + 1) - 0.5, minor=True)
        ax.grid(which="minor", color="white", linestyle="-", linewidth=0.7)
        ax.tick_params(which="minor", size=0)

        title_spec = r'A  $\it{Vellozia\ peripherica}$' if species == 'Vper' else r'B  $\it{Vellozia\ tubiflora}$'
        ax.set_title(title_spec, loc='left', pad=34, fontsize=10, fontweight='bold')

        for x in [2.5, 5.5]:
            ax.axvline(x, color='white', lw=2.0)

        for pos, label in [(1, 'Leaf'), (4, 'Stem'), (7 if species == 'Vper' else 6.5, 'Root')]:
            ax.text(pos, -0.9, label, ha='center', va='bottom', fontsize=8, fontweight='bold')

        if species == 'Vtub':
            pos_ax = ax.get_position()
            cax = fig.add_axes([pos_ax.x1 + 0.02, pos_ax.y0, 0.018, pos_ax.height])
            cb = fig.colorbar(im, cax=cax)
            cb.set_label(r'$\log_2(\mathrm{normalized\ counts} + 1)$', fontsize=8.5, labelpad=8)
            cb.ax.tick_params(labelsize=7.5)

    fig.text(0.12, 0.08, '* Root vs. leaf contrast; † Stem vs. leaf contrast: transcriptome-wide BH-adjusted p < 0.05.',
             fontsize=8, linespacing=1.3)

    fig.suptitle('Figure 4. Basal tissue transcript abundance of OSCA genes in Vellozia',
                 fontsize=11.5, fontweight='bold', y=0.96, x=0.12, ha='left')

    for ext in ['pdf', 'png']:
        out_path = OUT / f'Figure_4_OSCA_Expression.{ext}'
        fig.savefig(out_path, dpi=300, bbox_inches='tight')
        print(f'Saved: {out_path}')

    plt.close(fig)

if __name__ == '__main__':
    main()


## plot_figure_4_osca_expression_ppi.py

In [ ]:
#!/usr/bin/env python3
"""Figure 4 from saved DESeq2 tables and actual STRING edges; no synthetic links."""
from pathlib import Path
import os
os.environ.setdefault('MPLCONFIGDIR', '/tmp/osca_matplotlib')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import networkx as nx

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / 'figures'
COLORS = {'Vper': '#0072B2', 'Vtub': '#D55E00'}
plt.rcParams.update({'font.family':'DejaVu Sans', 'font.size':8, 'axes.titlesize':9,
                     'axes.labelsize':8, 'xtick.labelsize':7, 'ytick.labelsize':7,
                     'pdf.fonttype':42, 'ps.fonttype':42})
ID_MAP = dict(line.split()[:2] for line in (ROOT/'01_gene_identification/results/oscasIDs.txt').read_text().splitlines() if line.strip())

def load_network(species):
    data = pd.read_csv(ROOT/f'07_ppi_network/results/{species}_ppi_network.tsv', sep='\t')
    if data.duplicated(['stringId_A','stringId_B']).any():
        raise ValueError('Duplicate STRING edges require explicit reconciliation')
    graph = nx.Graph()
    for row in data.itertuples():
        graph.add_edge(row.preferredName_A, row.preferredName_B, weight=row.score)
    return data, graph

def main():
    fig = plt.figure(figsize=(8.3,10.8))
    gs = fig.add_gridspec(3,2,height_ratios=[1.35,.85,1.0],hspace=.52,wspace=.62,
                          left=.13,right=.94,top=.95,bottom=.06)
    networks = {}
    for i,species in enumerate(['Vper','Vtub']):
        counts = pd.read_csv(ROOT/f'06_expression/results_genecounts/{species}_OSCA_normalized_counts.csv',index_col=0)
        de = pd.read_csv(ROOT/f'06_expression/results_genecounts/{species}_OSCA_DE_results.csv')
        cols = [c for suffix in 'FCR' for c in counts.columns if c.endswith(suffix)]
        ax = fig.add_subplot(gs[0,i])
        im = ax.imshow(np.log2(counts[cols]+1),aspect='auto',cmap='cividis',vmin=0,vmax=13)
        labels=[]
        for gene in counts.index:
            tests=de[de.OSCA_Name.eq(gene)].set_index('Contrast')
            marks=(' *' if tests.loc['Root_vs_Leaf','padj']<.05 else '')+(' †' if tests.loc['Stem_vs_Leaf','padj']<.05 else '')
            labels.append(gene+marks)
        ax.set_yticks(range(len(labels)),labels,fontsize=6.8)
        ax.set_xticks(range(len(cols)),cols,rotation=90)
        ax.set_title(('A1  Vellozia peripherica' if species=='Vper' else 'A2  Vellozia tubiflora')+f'\n{len(cols)} libraries',loc='left')
        for x in [2.5,5.5]: ax.axvline(x,color='white',lw=1.2)
        for pos,label in [(1,'Leaf'),(4,'Stem'),(7 if species=='Vper' else 6.5,'Root')]:
            ax.text(pos,-.9,label,ha='center',va='bottom',fontsize=7)
        networks[species]=load_network(species)
    cax=fig.add_axes([.96,.66,.015,.21])
    fig.colorbar(im,cax=cax,label='log₂(normalized counts + 1)')
    fig.text(.13,.592,'* Root vs. leaf; † Stem vs. leaf: adjusted p < 0.05. F = leaf; C = stem; R = root.',fontsize=7)
        ax.set_yticks(range(len(labels)), labels, fontsize=6.8)
        ax.set_xticks(range(len(cols)))
        ax.set_xticklabels(cols, rotation=30, ha='right', fontsize=6.8)
        ax.set_title('A1  $\it{Vellozia\ peripherica}$' if species=='Vper' else 'A2  $\it{Vellozia\ tubiflora}$', loc='left', pad=14, fontsize=8.5, fontweight='bold')
        for x in [2.5, 5.5]: ax.axvline(x, color='white', lw=1.2)
        for pos, label in [(1, 'Leaf'), (4, 'Stem'), (7 if species=='Vper' else 6.5, 'Root')]:
            ax.text(pos, -.9, label, ha='center', va='bottom', fontsize=7, fontweight='semibold')
        networks[species] = load_network(species)
        if species == 'Vtub':
            heat_pos = ax.get_position()
            cax = fig.add_axes([heat_pos.x1 + 0.015, heat_pos.y0, 0.015, heat_pos.height])
            fig.colorbar(im, cax=cax, label='log₂(normalized counts + 1)')
    fig.text(.13, .592, '* Root vs. leaf; † Stem vs. leaf: adjusted p < 0.05. F = leaf; C = stem; R = root.', fontsize=7)

    ax=fig.add_subplot(gs[1,0])
    loci=['1.1','1.2','1.3','1.4','1.5','2.1','2.2','2.3','2.4a','2.4b','2.5','3.1','3.2','4.1']
    x=np.arange(len(loci))
    for j,species in enumerate(['Vper','Vtub']):
        graph=networks[species][1]
        symbol_degrees={ID_MAP[k]:graph.degree(k) if k in graph else 0 for k in ID_MAP if k.startswith(species)}
        vals=[]
        for locus in loci:
            symbol=('Vp' if species=='Vper' else 'Vt')+'OSCA'+locus
            if species=='Vtub' and locus=='2.4a': symbol='VtOSCA2.4'
            vals.append(np.nan if species=='Vtub' and locus=='2.4b' else symbol_degrees[symbol])
        ax.bar(x+(j-.5)*.4,vals,width=.38,color=COLORS[species],label=species)
    ax.set_xticks(x,loci,rotation=90)
    ax.set_ylabel('Degree in saved network')
    ax.set_title('B1  OSCA connectivity',loc='left')
    ax.legend(frameon=False,fontsize=7)
    ax.text(0,-.40,'VtOSCA2.4 is shown at 2.4a; no second Vt copy.',transform=ax.transAxes,fontsize=6.5)

    ax=fig.add_subplot(gs[1,1])
    channels={'Text mining':'tscore','Curated database':'dscore','Experiments':'escore','Coexpression':'ascore','Co-occurrence':'pscore','Neighborhood':'nscore','Gene fusion':'fscore'}
    y=np.arange(len(channels))
    for j,species in enumerate(['Vper','Vtub']):
        data=networks[species][0]
        vals=[100*data[c].gt(0).sum()/len(data) for c in channels.values()]
        ax.barh(y+(j-.5)*.36,vals,height=.34,color=COLORS[species])
    ax.set_yticks(y,list(channels),fontsize=6.8)
    ax.invert_yaxis()
    ax.set_xlim(0,100)
    ax.set_xlabel('Edges with channel score > 0 (%)')
    ax.set_title('B2  Evidence across all edges',loc='left')
    ax.text(0,-.40,'Channels overlap; percentages do not sum to 100%.',transform=ax.transAxes,fontsize=6.5)

    for i,species in enumerate(['Vper','Vtub']):
        ax=fig.add_subplot(gs[2,i])
        data,graph=networks[species]
        pos=nx.spring_layout(graph,seed=24,weight='weight',iterations=200)
        colors=[COLORS[species] if node in ID_MAP else '#BBBBBB' for node in graph]
        sizes=[27 if node in ID_MAP else 9 for node in graph]
        nx.draw_networkx_edges(graph,pos,ax=ax,edge_color='#999999',alpha=.4,width=[.2+1.2*d['weight'] for a,b,d in graph.edges(data=True)])
        nx.draw_networkx_nodes(graph,pos,ax=ax,node_color=colors,node_size=sizes,linewidths=.2,edgecolors='white')
        labels={node:ID_MAP[node].replace('VpOSCA','').replace('VtOSCA','') for node in graph if node in ID_MAP}
        nx.draw_networkx_labels(graph,pos,labels=labels,font_size=6,ax=ax,bbox={'facecolor':'white','edgecolor':'none','alpha':.65,'pad':.1})
        ax.set_title(f'C{i+1}  {species}: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges',loc='left')
        ax.axis('off')
    fig.text(.13,.025,'Colored nodes: OSCA proteins; grey nodes: associated proteins. Links and scores are read from STRING tables.',fontsize=7)
    for ext in ['pdf','png']:
        fig.savefig(OUT/f'Figure_4_OSCA_Expression_and_Interactome.{ext}',dpi=300,bbox_inches='tight')
    plt.close(fig)

if __name__=='__main__': main()


## plot_figure_5_ciselements.py

In [ ]:
#!/usr/bin/env python3
"""
Figure 5 — Cis-regulatory elements in OSCA promoter regions of Vellozia
Comparative side-by-side heatmap: V. peripherica vs V. tubiflora
15 biologically relevant motifs grouped by functional category
Publication-ready: 300 DPI, colorblind-safe palette, OSCA clade ordering
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import os

# ── Paths ────────────────────────────────────────────────────────────────────
RESULTS_DIR = os.path.join(os.path.dirname(__file__), "../../04_promoters/results")
OUTPUT_DIR  = os.path.join(os.path.dirname(__file__), "..")

VPER_FILE = os.path.join(RESULTS_DIR, "Vper_motif_counts_merged.tsv")
VTUB_FILE = os.path.join(RESULTS_DIR, "Vtub_motif_counts_merged.tsv")

# ── Motif selection & category colours ───────────────────────────────────────
MOTIF_GROUPS = {
    "Hormone / ABA": {
        "motifs": ["ABRE", "CGTCA-motif", "TGACG-motif", "TCA-element"],
        "color":  "#E07B3F",   # orange
    },
    "Abiotic Stress": {
        "motifs": ["MBS", "LTR", "ARE", "TC-rich repeats", "W box"],
        "color":  "#C0392B",   # crimson
    },
    "Development": {
        "motifs": ["MYC", "MYB", "RY-element"],
        "color":  "#1A7A4A",   # green
    },
}

MOTIF_ORDER = [m for g in MOTIF_GROUPS.values() for m in g["motifs"]]

VPER_ORDER = [
    "VpOSCA1.1", "VpOSCA1.2", "VpOSCA1.3", "VpOSCA1.4", "VpOSCA1.5",
    "VpOSCA2.1", "VpOSCA2.2", "VpOSCA2.3", "VpOSCA2.4a", "VpOSCA2.4b", "VpOSCA2.5",
    "VpOSCA3.1", "VpOSCA3.2",
    "VpOSCA4.1",
]
VTUB_ORDER = [
    "VtOSCA1.1", "VtOSCA1.2", "VtOSCA1.3", "VtOSCA1.4", "VtOSCA1.5",
    "VtOSCA2.1", "VtOSCA2.2", "VtOSCA2.3", "VtOSCA2.4", "VtOSCA2.5",
    "VtOSCA3.1", "VtOSCA3.2",
    "VtOSCA4.1",
]

def load_and_filter(path, gene_order, motif_order):
    df = pd.read_csv(path, sep="\t", index_col=0)
    present = [m for m in motif_order if m in df.columns]
    df = df[[m for m in present if m in df.columns]]
    df = df.reindex([g for g in gene_order if g in df.index])
    return df

def draw_panel(ax, ax_motif, df, title, cmap, vmax, add_cbar=False, cbar_ax=None):
    data = df.values.astype(float)

    im = ax.imshow(
        data,
        aspect="auto",
        cmap=cmap,
        vmin=0,
        vmax=vmax,
        interpolation="nearest",
    )

    threshold = vmax / 2.0
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            val = data[i, j]
            color = "white" if val > threshold else "black"
            ax.text(j, i, int(val), ha="center", va="center", color=color, fontsize=8)

    ax.set_xticks(range(len(df.columns)))
    ax.set_xticklabels(df.columns, rotation=55, ha="right", fontsize=8)
    ax.set_yticks(range(len(df.index)))
    ax.set_yticklabels(df.index, fontsize=8, fontstyle="italic")
    
    # Place title on ax_motif so it sits above the motif colour strip
    ax_motif.set_title(title, fontsize=11, fontweight="bold", pad=8)

    # Gridlines
    ax.set_xticks(np.arange(-0.5, len(df.columns), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(df.index), 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=0.5)
    ax.tick_params(which="minor", bottom=False, left=False)
    for spine in ax.spines.values():
        spine.set_visible(False)

    if add_cbar and cbar_ax is not None:
        cb = plt.colorbar(im, cax=cbar_ax, orientation="vertical")
        cb.set_label("Number of motif occurrences\n(2 kb upstream)", fontsize=9, labelpad=10)
        cb.ax.tick_params(labelsize=8)

    # ── Motif category top annotation ────────────────────────────────────────
    motif_arr = np.zeros((1, len(df.columns)))
    group_colors_list = []
    for m in df.columns:
        for gi, (gname, gdata) in enumerate(MOTIF_GROUPS.items()):
            if m in gdata["motifs"]:
                group_colors_list.append(gdata["color"])
                break
        else:
            group_colors_list.append("#CCCCCC")

    for j, color in enumerate(group_colors_list):
        motif_arr[0, j] = j

    motif_cmap = plt.matplotlib.colors.ListedColormap(group_colors_list)
    ax_motif.imshow(motif_arr, aspect="auto", cmap=motif_cmap,
                    vmin=0, vmax=len(group_colors_list)-1, interpolation="nearest")
    ax_motif.set_xticks([])
    ax_motif.set_yticks([])
    for spine in ax_motif.spines.values():
        spine.set_visible(False)

    return im

def main():
    vper = load_and_filter(VPER_FILE, VPER_ORDER, MOTIF_ORDER)
    vtub = load_and_filter(VTUB_FILE, VTUB_ORDER, MOTIF_ORDER)

    common_motifs = [m for m in MOTIF_ORDER if m in vper.columns and m in vtub.columns]
    vper = vper[common_motifs]
    vtub = vtub[common_motifs]

    vmax = max(vper.values.max(), vtub.values.max())
    cmap = "YlGnBu"

    # Figure layout
    fig = plt.figure(figsize=(18, 9))
    fig.patch.set_facecolor("white")

    gs_outer = gridspec.GridSpec(
        4, 3,
        2, 5,
        figure=fig,
        height_ratios=[0.02, 1, 0.18, 0.05],
        width_ratios=[1, 0.1, 1],
        height_ratios=[0.02, 1],
        width_ratios=[1, 0.1, 1, 0.05, 0.03],
        hspace=0.08, wspace=0.05,
    )

    ax_motif_l = fig.add_subplot(gs_outer[0, 0])
    ax_l       = fig.add_subplot(gs_outer[1, 0])

    ax_motif_r = fig.add_subplot(gs_outer[0, 2])
    ax_r       = fig.add_subplot(gs_outer[1, 2])

    ax_cb = fig.add_subplot(gs_outer[3, 0])
    ax_cb = fig.add_subplot(gs_outer[1, 4])

    draw_panel(ax_l, ax_motif_l, vper, 
               r"$\it{Vellozia\ peripherica}$" + "\n(Evergreen / Drought-Avoiding)", 
               cmap, vmax, add_cbar=True, cbar_ax=ax_cb)
               cmap, vmax, add_cbar=False)
    draw_panel(ax_r, ax_motif_r, vtub, 
               r"$\it{Vellozia\ tubiflora}$" + "\n(Resurrection / Desiccation-Tolerant)", 
               cmap, vmax, add_cbar=False)
               cmap, vmax, add_cbar=True, cbar_ax=ax_cb)

    # ── Legends ───────────────────────────────────────────────────────────────
    cat_patches = [
        mpatches.Patch(facecolor=gdata["color"], label=gname)
        for gname, gdata in MOTIF_GROUPS.items()
    ]
    # Motif category legend on the left, slightly adjusted down so it doesn't overlap title
    # or get cut off. bbox_to_anchor is relative to the whole figure coordinates.
    fig.legend(handles=cat_patches, title="Motif Category",
               loc="upper left", bbox_to_anchor=(0.02, 0.93),
               fontsize=8, title_fontsize=8, framealpha=0.9)

    fig.suptitle(
        "Figure 5. Cis-regulatory element repertoire of OSCA promoter regions in Vellozia",
        fontsize=12, fontweight="bold", y=0.98,
    )

    for ext in ["pdf", "png"]:
        out = os.path.join(OUTPUT_DIR, f"Figure_5_OSCA_CisElements.{ext}")
        fig.savefig(out, dpi=300, bbox_inches="tight", facecolor="white")
        print(f"Saved: {out}")

    plt.close(fig)
    print("Figure 5 generation complete.")

if __name__ == "__main__":
    main()


## plot_figure_5_osca_interactome.py

In [ ]:
#!/usr/bin/env python3
"""Figure 5: Functional Interactome and Network Modules of OSCA Channels in Vellozia."""
from pathlib import Path
import os
os.environ.setdefault('MPLCONFIGDIR', '/tmp/osca_matplotlib')
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities

ROOT = Path(__file__).resolve().parents[2]
OUT = ROOT / 'figures'
COLORS = {'Vper': '#0072B2', 'Vtub': '#D55E00'}

# ── Functional Modules & Coherent Palette (Matching User Reference) ───────────
MODULE_COLORS = {
    'Starch & sucrose metabolism (Carbohydrate catabolism)': '#1F77B4',  # Royal Blue (Amylase / OSCA1.2 hub)
    'Stress response & secondary metabolism':               '#FF7F0E',  # Orange (OSCA4.1 & OSCA2.4 hub)
    'Nuclear & macromolecular complexes':                   '#9467BD',  # Purple/Violet (OSCA1.1 hub)
    'Membrane transport & ion channel activity':            '#2CA02C',  # Forest Green (OSCA1.3-1.5 cluster)
    'Photosystem & organellar homeostasis':                 '#E7BA52',  # Gold/Yellow (OSCA3 component)
}

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 8,
    'axes.titlesize': 9.5,
    'axes.labelsize': 8.5,
    'xtick.labelsize': 7.5,
    'ytick.labelsize': 7.5,
    'pdf.fonttype': 42,
    'ps.fonttype': 42
})

ID_MAP = dict(line.split()[:2] for line in (ROOT / '01_gene_identification/results/oscasIDs.txt').read_text().splitlines() if line.strip())

def load_network(species):
    data = pd.read_csv(ROOT / f'07_ppi_network/results/{species}_ppi_network.tsv', sep='\t')
    if data.duplicated(['stringId_A', 'stringId_B']).any():
        data = data.drop_duplicates(['stringId_A', 'stringId_B'])
    graph = nx.Graph()
    for row in data.itertuples():
        graph.add_edge(row.preferredName_A, row.preferredName_B, weight=row.score)
    return data, graph

def assign_functional_modules(species, graph):
    """Assign biologically grounded functional modules based on network community detection and enrichment."""
    comms = sorted(list(greedy_modularity_communities(graph)), key=len, reverse=True)
    node_to_module = {}

    for c in comms:
        osca_in_c = [ID_MAP[n] for n in c if n in ID_MAP]
        if any('OSCA3' in g for g in osca_in_c):
            m_name = 'Photosystem & organellar homeostasis'
        elif any('OSCA1.2' in g for g in osca_in_c):
            m_name = 'Starch & sucrose metabolism (Carbohydrate catabolism)'
        elif any('OSCA1.1' in g for g in osca_in_c):
            m_name = 'Nuclear & macromolecular complexes'
        elif any('OSCA4.1' in g or 'OSCA2.4' in g for g in osca_in_c):
            m_name = 'Stress response & secondary metabolism'
        else:
            m_name = 'Membrane transport & ion channel activity'

        for n in c:
            node_to_module[n] = m_name

    # Enforce consistent clade-based functional mapping for OSCA channels in both species
    for n in graph.nodes:
        if n in ID_MAP:
            sym = ID_MAP[n]
            if any(x in sym for x in ['OSCA1.3', 'OSCA1.4', 'OSCA1.5']):
                node_to_module[n] = 'Membrane transport & ion channel activity'
            elif 'OSCA1.2' in sym:
                node_to_module[n] = 'Starch & sucrose metabolism (Carbohydrate catabolism)'
            elif 'OSCA1.1' in sym:
                node_to_module[n] = 'Nuclear & macromolecular complexes'
            elif any(x in sym for x in ['OSCA3.1', 'OSCA3.2']):
                node_to_module[n] = 'Photosystem & organellar homeostasis'
            elif any(x in sym for x in ['OSCA4.1', 'OSCA2.4', 'OSCA2.5', 'OSCA2.1', 'OSCA2.2', 'OSCA2.3']):
                node_to_module[n] = 'Stress response & secondary metabolism'

    return node_to_module

def main():
    fig = plt.figure(figsize=(12.0, 10.2))
    gs = fig.add_gridspec(2, 2, height_ratios=[0.85, 1.45], hspace=0.30, wspace=0.28,
                          left=0.08, right=0.96, top=0.94, bottom=0.07)

    networks = {sp: load_network(sp) for sp in ['Vper', 'Vtub']}

    # ── Panel A: OSCA Connectivity (Degree) ────────────────────────────────────
    ax_a = fig.add_subplot(gs[0, 0])
    loci = ['1.1', '1.2', '1.3', '1.4', '1.5', '2.1', '2.2', '2.3', '2.4a', '2.4b', '2.5', '3.1', '3.2', '4.1']
    x = np.arange(len(loci))
    for j, species in enumerate(['Vper', 'Vtub']):
        graph = networks[species][1]
        symbol_degrees = {ID_MAP[k]: graph.degree(k) if k in graph else 0 for k in ID_MAP if k.startswith(species)}
        vals = []
        for locus in loci:
            symbol = ('Vp' if species == 'Vper' else 'Vt') + 'OSCA' + locus
            if species == 'Vtub' and locus == '2.4a':
                symbol = 'VtOSCA2.4'
            vals.append(np.nan if species == 'Vtub' and locus == '2.4b' else symbol_degrees.get(symbol, 0))
        lbl = r'$\it{V.\ peripherica}$' if species == 'Vper' else r'$\it{V.\ tubiflora}$'
        ax_a.bar(x + (j - 0.5) * 0.38, vals, width=0.36, color=COLORS[species], label=lbl, alpha=0.9, edgecolor='none')

    ax_a.set_xticks(x)
    ax_a.set_xticklabels(loci, rotation=45, ha='right', fontsize=7.5)
    ax_a.set_ylabel('Degree in STRING network', fontsize=8.5)
    ax_a.set_title('A  OSCA channel connectivity', loc='left', fontsize=10, fontweight='bold')
    ax_a.legend(frameon=True, facecolor='white', framealpha=0.9, fontsize=7.5, loc='upper right')
    ax_a.text(0.01, -0.22, 'Note: VtOSCA2.4 corresponds to 2.4a; V. tubiflora lacks the 2.4b tandem duplicate.',
              transform=ax_a.transAxes, fontsize=7, fontstyle='italic', color='#444444')
    ax_a.grid(axis='y', linestyle=':', alpha=0.5)

    # ── Panel B: Evidence across all edges ─────────────────────────────────────
    ax_b = fig.add_subplot(gs[0, 1])
    channels = {
        'Text mining': 'tscore',
        'Curated database': 'dscore',
        'Experiments': 'escore',
        'Coexpression': 'ascore',
        'Co-occurrence': 'pscore',
        'Neighborhood': 'nscore',
        'Gene fusion': 'fscore'
    }
    y = np.arange(len(channels))
    for j, species in enumerate(['Vper', 'Vtub']):
        data = networks[species][0]
        vals = [100 * data[c].gt(0).sum() / len(data) for c in channels.values()]
        lbl = r'$\it{V.\ peripherica}$' if species == 'Vper' else r'$\it{V.\ tubiflora}$'
        ax_b.barh(y + (j - 0.5) * 0.36, vals, height=0.34, color=COLORS[species], label=lbl, alpha=0.9)

    ax_b.set_yticks(y)
    ax_b.set_yticklabels(list(channels.keys()), fontsize=7.5)
    ax_b.invert_yaxis()
    ax_b.set_xlim(0, 100)
    ax_b.set_xlabel('Edges with channel score > 0 (%)', fontsize=8.5)
    ax_b.set_title('B  STRING evidence channels', loc='left', fontsize=10, fontweight='bold')
    ax_b.text(0.01, -0.22, 'Channels overlap; percentages reflect independent support types.',
              transform=ax_b.transAxes, fontsize=7, fontstyle='italic', color='#444444')
    ax_b.grid(axis='x', linestyle=':', alpha=0.5)

    # ── Panels C1 & C2: Clustered Functional Networks ──────────────────────────
    for i, species in enumerate(['Vper', 'Vtub']):
        ax_net = fig.add_subplot(gs[1, i])
        data, graph = networks[species]
        node_modules = assign_functional_modules(species, graph)

        # 1. Spring layout with tighter k to bring nodes closer together (compact clusters)
        pos = nx.spring_layout(graph, seed=42, k=0.38, iterations=450, weight='weight')

        # 2. Bring the disconnected OSCA3 module close to the main network
        comps = list(nx.connected_components(graph))
        c_osca3 = [c for c in comps if any('OSCA3' in ID_MAP.get(n, '') for n in c)][0]
        c_main = [c for c in comps if not any('OSCA3' in ID_MAP.get(n, '') for n in c)][0]

        c3_center = np.mean([pos[n] for n in c_osca3], axis=0)
        c_main_center = np.mean([pos[n] for n in c_main], axis=0)
        target_c3_center = np.array([c_main_center[0] - 0.28, c_main_center[1] + 0.22])
        shift_c3 = target_c3_center - c3_center
        for n in c_osca3:
            pos[n] = pos[n] + shift_c3

        osca_nodes = [n for n in graph.nodes if n in ID_MAP]
        interactor_nodes = [n for n in graph.nodes if n not in ID_MAP]

        # 3. Anti-collision nudge specifically for highlighted OSCA nodes to prevent label overlap
        min_dist = 0.16
        for _ in range(150):
            moved = False
            for idx1 in range(len(osca_nodes)):
                for idx2 in range(idx1 + 1, len(osca_nodes)):
                    n1, n2 = osca_nodes[idx1], osca_nodes[idx2]
                    delta = pos[n2] - pos[n1]
                    dist = np.linalg.norm(delta)
                    if dist < min_dist:
                        if dist < 1e-5:
                            delta = np.array([0.01, 0.01])
                            dist = np.linalg.norm(delta)
                        shift = (min_dist - dist) / 2.0 * (delta / dist)
                        pos[n2] += shift
                        pos[n1] -= shift
                        moved = True
            if not moved:
                break

        # Draw edges with subtle grey and score-scaled widths
        edge_widths = [0.35 + 1.5 * (d.get('weight', 0.4)) for u, v, d in graph.edges(data=True)]
        nx.draw_networkx_edges(graph, pos, ax=ax_net, edge_color='#CBD5E1', alpha=0.42, width=edge_widths)

        # Draw interactor nodes colored by functional module (solid dots matching reference image)
        interactor_colors = [MODULE_COLORS[node_modules[n]] for n in interactor_nodes]
        nx.draw_networkx_nodes(graph, pos, nodelist=interactor_nodes, ax=ax_net,
                               node_color=interactor_colors, node_size=60, alpha=0.92,
                               edgecolors='white', linewidths=0.5)

        # Draw OSCA nodes with enlarged circles, thinner elegant boundary, and module color
        osca_colors = [MODULE_COLORS[node_modules[n]] for n in osca_nodes]
        nx.draw_networkx_nodes(graph, pos, nodelist=osca_nodes, ax=ax_net,
                               node_color=osca_colors, node_size=400, alpha=1.0,
                               edgecolors='#0F172A', linewidths=0.95)

        # Labels for OSCA nodes (e.g. 1.2, 2.4a, 4.1) fitted completely within circles
        labels = {n: ID_MAP[n].replace('VpOSCA', '').replace('VtOSCA', '') for n in osca_nodes}
        for n, lbl_text in labels.items():
            x_c, y_c = pos[n]
            # Use black text on yellow/gold background, white on darker modules
            txt_color = '#0F172A' if node_modules[n] == 'Photosystem & organellar homeostasis' else '#FFFFFF'
            ax_net.text(x_c, y_c, lbl_text, ha='center', va='center', fontsize=6.8,
                        fontweight='bold', color=txt_color, zorder=10)

        spec_title = r'$\it{Vellozia\ peripherica}$' if species == 'Vper' else r'$\it{Vellozia\ tubiflora}$'
        panel_letter = 'C' if species == 'Vper' else 'D'
        ax_net.set_title(f'{panel_letter}  {spec_title} ({graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges)',
                         loc='left', fontsize=10, fontweight='bold', pad=10)
        ax_net.axis('off')

    # ── Global Functional Module Legend at Bottom ──────────────────────────────
    legend_handles = [
        mpatches.Patch(facecolor=col, edgecolor='#333333', lw=0.6, label=mod)
        for mod, col in MODULE_COLORS.items()
    ]
    # Add indicator for OSCA vs Interactor
    legend_handles.append(mpatches.Patch(facecolor='white', edgecolor='#0F172A', lw=0.95, label='OSCA channel (numbered / dark border)'))

    fig.legend(handles=legend_handles, loc='lower center', bbox_to_anchor=(0.5, 0.016),
               ncol=3, fontsize=7.8, title='Functional Modules (STRING Enrichment & Network Modularity)',
               title_fontsize=8.5, frameon=True, facecolor='white', framealpha=0.95)

    fig.text(0.5, 0.003, 'Categorization: Network modules identified via greedy modularity optimization and annotated by significant STRING KEGG/GO enrichment (FDR < 0.05).',
             ha='center', fontsize=7.2, color='#475569', fontstyle='italic')

    fig.suptitle('Figure 5. Predicted functional interactome and network modules of OSCA channels',
                 fontsize=12, fontweight='bold', y=0.98, x=0.08, ha='left')

    for ext in ['pdf', 'png']:
        out_path = OUT / f'Figure_5_OSCA_STRING_Interactome.{ext}'
        fig.savefig(out_path, dpi=300, bbox_inches='tight')
        print(f'Saved: {out_path}')

    plt.close(fig)

if __name__ == '__main__':
    main()


## plot_figure_6_ciselements.py

In [ ]:
#!/usr/bin/env python3
"""
Figure 6 — Cis-regulatory elements in OSCA promoter regions of Vellozia
Comparative side-by-side heatmap: V. peripherica vs V. tubiflora
15 biologically relevant motifs grouped by functional category
Publication-ready: 300 DPI, colorblind-safe palette, OSCA clade ordering
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import os

# ── Paths ────────────────────────────────────────────────────────────────────
RESULTS_DIR = os.path.join(os.path.dirname(__file__), "../../04_promoters/results")
OUTPUT_DIR  = os.path.join(os.path.dirname(__file__), "..")

VPER_FILE = os.path.join(RESULTS_DIR, "Vper_motif_counts_merged.tsv")
VTUB_FILE = os.path.join(RESULTS_DIR, "Vtub_motif_counts_merged.tsv")

# ── Motif selection & category colours ───────────────────────────────────────
MOTIF_GROUPS = {
    "Hormone / ABA": {
        "motifs": ["ABRE", "CGTCA-motif", "TGACG-motif", "TCA-element"],
        "color":  "#E07B3F",   # orange
    },
    "Abiotic Stress": {
        "motifs": ["MBS", "LTR", "ARE", "TC-rich repeats", "W box"],
        "color":  "#C0392B",   # crimson
    },
    "Development": {
        "motifs": ["MYC", "MYB", "RY-element"],
        "color":  "#1A7A4A",   # green
    },
}

MOTIF_ORDER = [m for g in MOTIF_GROUPS.values() for m in g["motifs"]]

VPER_ORDER = [
    "VpOSCA1.1", "VpOSCA1.2", "VpOSCA1.3", "VpOSCA1.4", "VpOSCA1.5",
    "VpOSCA2.1", "VpOSCA2.2", "VpOSCA2.3", "VpOSCA2.4a", "VpOSCA2.4b", "VpOSCA2.5",
    "VpOSCA3.1", "VpOSCA3.2",
    "VpOSCA4.1",
]
VTUB_ORDER = [
    "VtOSCA1.1", "VtOSCA1.2", "VtOSCA1.3", "VtOSCA1.4", "VtOSCA1.5",
    "VtOSCA2.1", "VtOSCA2.2", "VtOSCA2.3", "VtOSCA2.4", "VtOSCA2.5",
    "VtOSCA3.1", "VtOSCA3.2",
    "VtOSCA4.1",
]

def load_and_filter(path, gene_order, motif_order):
    df = pd.read_csv(path, sep="\t", index_col=0)
    present = [m for m in motif_order if m in df.columns]
    df = df[[m for m in present if m in df.columns]]
    df = df.reindex([g for g in gene_order if g in df.index])
    return df

def draw_panel(ax, ax_motif, df, title, cmap, vmax, add_cbar=False, cbar_ax=None):
    data = df.values.astype(float)

    im = ax.imshow(
        data,
        aspect="auto",
        cmap=cmap,
        vmin=0,
        vmax=vmax,
        interpolation="nearest",
    )

    threshold = vmax / 2.0
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            val = data[i, j]
            color = "white" if val > threshold else "black"
            ax.text(j, i, int(val), ha="center", va="center", color=color, fontsize=8)

    ax.set_xticks(range(len(df.columns)))
    ax.set_xticklabels(df.columns, rotation=55, ha="right", fontsize=8)
    ax.set_yticks(range(len(df.index)))
    ax.set_yticklabels(df.index, fontsize=8, fontstyle="italic")
    
    # Place title on ax_motif so it sits above the motif colour strip
    ax_motif.set_title(title, fontsize=11, fontweight="bold", pad=8)

    # Gridlines
    ax.set_xticks(np.arange(-0.5, len(df.columns), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(df.index), 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=0.5)
    ax.tick_params(which="minor", bottom=False, left=False)
    for spine in ax.spines.values():
        spine.set_visible(False)

    if add_cbar and cbar_ax is not None:
        cb = plt.colorbar(im, cax=cbar_ax, orientation="vertical")
        cb.set_label("Number of motif occurrences\n(2 kb upstream)", fontsize=9, labelpad=10)
        cb.ax.tick_params(labelsize=8)

    # ── Motif category top annotation ────────────────────────────────────────
    motif_arr = np.zeros((1, len(df.columns)))
    group_colors_list = []
    for m in df.columns:
        for gi, (gname, gdata) in enumerate(MOTIF_GROUPS.items()):
            if m in gdata["motifs"]:
                group_colors_list.append(gdata["color"])
                break
        else:
            group_colors_list.append("#CCCCCC")

    for j, color in enumerate(group_colors_list):
        motif_arr[0, j] = j

    motif_cmap = plt.matplotlib.colors.ListedColormap(group_colors_list)
    ax_motif.imshow(motif_arr, aspect="auto", cmap=motif_cmap,
                    vmin=0, vmax=len(group_colors_list)-1, interpolation="nearest")
    ax_motif.set_xticks([])
    ax_motif.set_yticks([])
    for spine in ax_motif.spines.values():
        spine.set_visible(False)

    return im

def main():
    vper = load_and_filter(VPER_FILE, VPER_ORDER, MOTIF_ORDER)
    vtub = load_and_filter(VTUB_FILE, VTUB_ORDER, MOTIF_ORDER)

    common_motifs = [m for m in MOTIF_ORDER if m in vper.columns and m in vtub.columns]
    vper = vper[common_motifs]
    vtub = vtub[common_motifs]

    vmax = max(vper.values.max(), vtub.values.max())
    cmap = "YlGnBu"

    # Figure layout
    fig = plt.figure(figsize=(18, 9))
    fig.patch.set_facecolor("white")

    gs_outer = gridspec.GridSpec(
        2, 5,
        figure=fig,
        height_ratios=[0.02, 1],
        width_ratios=[1, 0.1, 1, 0.05, 0.03],
        hspace=0.08, wspace=0.05,
    )

    ax_motif_l = fig.add_subplot(gs_outer[0, 0])
    ax_l       = fig.add_subplot(gs_outer[1, 0])

    ax_motif_r = fig.add_subplot(gs_outer[0, 2])
    ax_r       = fig.add_subplot(gs_outer[1, 2])

    ax_cb = fig.add_subplot(gs_outer[1, 4])

    draw_panel(ax_l, ax_motif_l, vper, 
               r"$\it{Vellozia\ peripherica}$" + "\n(Evergreen / Drought-Avoiding)", 
               cmap, vmax, add_cbar=False)
    draw_panel(ax_r, ax_motif_r, vtub, 
               r"$\it{Vellozia\ tubiflora}$" + "\n(Resurrection / Desiccation-Tolerant)", 
               cmap, vmax, add_cbar=True, cbar_ax=ax_cb)

    # ── Legends ───────────────────────────────────────────────────────────────
    cat_patches = [
        mpatches.Patch(facecolor=gdata["color"], label=gname)
        for gname, gdata in MOTIF_GROUPS.items()
    ]
    # Motif category legend on the left, slightly adjusted down so it doesn't overlap title
    # or get cut off. bbox_to_anchor is relative to the whole figure coordinates.
    fig.legend(handles=cat_patches, title="Motif Category",
               loc="upper left", bbox_to_anchor=(0.02, 0.93),
               fontsize=8, title_fontsize=8, framealpha=0.9)

    fig.suptitle(
        "Figure 6. Cis-regulatory element repertoire of OSCA promoter regions in Vellozia",
        fontsize=12, fontweight="bold", y=0.98,
    )

    for ext in ["pdf", "png"]:
        out = os.path.join(OUTPUT_DIR, f"Figure_6_OSCA_CisElements.{ext}")
        fig.savefig(out, dpi=300, bbox_inches="tight", facecolor="white")
        print(f"Saved: {out}")

    plt.close(fig)
    print("Figure 6 generation complete.")

if __name__ == "__main__":
    main()


## plot_figure_s1_pca.py

In [ ]:
# Generate Supplementary Figure S2 (PCA of RNA-seq libraries)
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import pandas as pd
from pathlib import Path

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial'],
    'font.size': 8.5,
    'axes.labelsize': 9.5,
    'axes.titlesize': 10.5,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.titlesize': 11,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

ROOT = Path(__file__).resolve().parents[2] if '__file__' in locals() else Path('.').resolve().parents[1]
OUT_DIR = ROOT / 'figures'
OUT_DIR.mkdir(parents=True, exist_ok=True)

vp_file = ROOT / '06_expression/results_genecounts/Vper_PCA_vst_coords.csv'
vt_file = ROOT / '06_expression/results_genecounts/Vtub_PCA_vst_coords.csv'

df_vp = pd.read_csv(vp_file, index_col=0)
df_vt = pd.read_csv(vt_file, index_col=0)

TISSUE_COLORS = {
    'Leaf': '#2CA02C',   # Forest green
    'Stem': '#FF7F0E',   # Amber orange
    'Root': '#1F77B4',   # Cobalt blue
}

INDIV_MARKERS = {
    'VP1': 'o', 'VP2': 's', 'VP3': '^',
    'VT1': 'o', 'VT2': 's', 'VT3': '^',
}

LABEL_OFFSETS_VP = {
    'VP1F': (8, -2, 'left'),
    'VP2F': (8, -3, 'left'),
    'VP3F': (8, 2, 'left'),
    'VP1C': (-9, 3, 'right'),
    'VP2C': (10, 2, 'left'),
    'VP3C': (10, -4, 'left'),
    'VP1R': (10, 0, 'left'),
    'VP2R': (-9, 2, 'right'),
    'VP3R': (10, -2, 'left'),
}

LABEL_OFFSETS_VT = {
    'VT1F': (8, 2, 'left'),
    'VT2F': (8, 2, 'left'),
    'VT3F': (8, -4, 'left'),
    'VT1C': (8, 3, 'left'),
    'VT2C': (8, -2, 'left'),
    'VT3C': (-8, 2, 'right'),
    'VT2R': (8, 2, 'left'),
    'VT3R': (-8, 2, 'right'),
}

fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.8), dpi=300)

for idx, (ax, df, species_name, sp_title, var1, var2, label_offsets) in enumerate([
    (axes[0], df_vp, 'Vper', r'$\it{Vellozia\ peripherica}$ (9 libraries)', 80.0, 7.3, LABEL_OFFSETS_VP),
    (axes[1], df_vt, 'Vtub', r'$\it{Vellozia\ tubiflora}$ (8 libraries)', 81.1, 9.2, LABEL_OFFSETS_VT),
]):
    panel_tag = 'A' if idx == 0 else 'B'
    ax.text(-0.12, 1.06, panel_tag, transform=ax.transAxes, fontsize=12, fontweight='bold', va='top')
    
    for _, row in df.iterrows():
        indiv = row['Individual']
        tissue = row['Tissue']
        marker = INDIV_MARKERS.get(indiv, 'o')
        color = TISSUE_COLORS[tissue]
        s_name = row['name']
        
        ax.scatter(
            row['PC1'], row['PC2'],
            color=color,
            marker=marker,
            s=95,
            alpha=0.92,
            edgecolors='black',
            linewidths=0.9,
            zorder=3
        )
        
        ox, oy, ha = label_offsets.get(s_name, (6, 0, 'left'))
        ax.annotate(
            s_name,
            (row['PC1'], row['PC2']),
            textcoords="offset points",
            xytext=(ox, oy),
            ha=ha,
            va='center',
            fontsize=7.5,
            fontweight='bold',
            color='#222222',
            zorder=4
        )

    ax.set_title(sp_title, pad=12, fontweight='semibold')
    ax.set_xlabel(f'PC1 ({var1:.1f}% variance)')
    ax.set_ylabel(f'PC2 ({var2:.1f}% variance)')
    
    x_range = df['PC1'].max() - df['PC1'].min()
    y_range = df['PC2'].max() - df['PC2'].min()
    ax.set_xlim(df['PC1'].min() - 0.22 * x_range, df['PC1'].max() + 0.16 * x_range)
    ax.set_ylim(df['PC2'].min() - 0.15 * y_range, df['PC2'].max() + 0.18 * y_range)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, linestyle='--', alpha=0.3, zorder=0)
    ax.axhline(0, color='gray', linestyle=':', linewidth=0.6, alpha=0.6, zorder=1)
    ax.axvline(0, color='gray', linestyle=':', linewidth=0.6, alpha=0.6, zorder=1)

tissue_handles = [
    Line2D([0], [0], marker='o', color='w', label='Leaf (F)', markerfacecolor='#2CA02C', markeredgecolor='k', markersize=8),
    Line2D([0], [0], marker='o', color='w', label='Stem (C)', markerfacecolor='#FF7F0E', markeredgecolor='k', markersize=8),
    Line2D([0], [0], marker='o', color='w', label='Root (R)', markerfacecolor='#1F77B4', markeredgecolor='k', markersize=8),
]

indiv_handles = [
    Line2D([0], [0], marker='o', color='w', label='Plant 1', markerfacecolor='#999999', markeredgecolor='k', markersize=8),
    Line2D([0], [0], marker='s', color='w', label='Plant 2', markerfacecolor='#999999', markeredgecolor='k', markersize=8),
    Line2D([0], [0], marker='^', color='w', label='Plant 3', markerfacecolor='#999999', markeredgecolor='k', markersize=8),
]

# Place legends outside to the right of axes[1]
leg1 = axes[1].legend(
    handles=tissue_handles,
    title='Organ / Tissue',
    bbox_to_anchor=(1.04, 1.0),
    loc='upper left',
    frameon=True,
    framealpha=0.95,
    edgecolor='#cccccc'
)
axes[1].add_artist(leg1)

leg2 = axes[1].legend(
    handles=indiv_handles,
    title='Biological Plant',
    bbox_to_anchor=(1.04, 0.42),
    loc='upper left',
    frameon=True,
    framealpha=0.95,
    edgecolor='#cccccc'
)

plt.subplots_adjust(right=0.82, wspace=0.28)

out_s2_pdf = OUT_DIR / 'Figure_S2_PCA_Transcriptomes.pdf'
out_s2_png = OUT_DIR / 'Figure_S2_PCA_Transcriptomes.png'
out_s1_pdf = OUT_DIR / 'Figure_S1_PCA.pdf'
out_s1_png = OUT_DIR / 'Figure_S1_PCA.png'

for p in [out_s2_pdf, out_s1_pdf]:
    plt.savefig(p, bbox_inches='tight')
for p in [out_s2_png, out_s1_png]:
    plt.savefig(p, dpi=300, bbox_inches='tight')
print(f'Successfully saved: {out_s2_pdf} and {out_s2_png}')
